In [2]:
from torch.quantization import quantize_dynamic
from sentence_transformers import SentenceTransformer, util, InputExample, losses
import pandas as pd

In [3]:
from sentence_transformers import SentenceTransformer, util
import numpy as np
from sklearn.cluster import AgglomerativeClustering
import seaborn as sns
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.cluster import MiniBatchKMeans
import gensim
from gensim import corpora
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, silhouette_score
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/mariannebenard/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
# Load SBERT model
model = SentenceTransformer('paraphrase-MiniLM-L3-v2')

In [5]:
import nltk

nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/mariannebenard/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [6]:
# Function to read and preprocess text data
def read_and_preprocess(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        text = file.read()
        text=text.replace('\n','')
    sentences = nltk.sent_tokenize(text)  # Tokenize into sentences
    return [sentence.strip() for sentence in sentences if sentence.strip()]

In [11]:
from tqdm import tqdm
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory
from langchain_ollama import OllamaLLM
llm = OllamaLLM(model="llama3.2")

for decade in tqdm(np.arange(1900, 2010, 10)):
  # Load datasets
  history_sentences = read_and_preprocess(f'src/data/wikipedia_timeline/{decade}s.txt')
  movies_sentences = read_and_preprocess(f'src/data/movies_summaries/summaries_decade_{decade}.txt')

  # Encode sentences into embeddings
  history_embeddings = model.encode(history_sentences, batch_size=32, convert_to_tensor=True)
  movies_embeddings = model.encode(movies_sentences, batch_size=32, convert_to_tensor=True)

  # Compute pairwise similarity matrix
  similarity_matrix = util.pytorch_cos_sim(history_embeddings, movies_embeddings).cpu().numpy()

  # Perform clustering on history dataset
  cluster_model = MiniBatchKMeans(n_clusters=5)
  history_clusters = cluster_model.fit_predict(history_embeddings.cpu().numpy())

  # Cluster sentences from history into semantic fields
  clustered_history_string=''
  clustered_history = {i: [] for i in range(max(history_clusters) + 1)}
  for idx, cluster_id in enumerate(history_clusters):
    clustered_history[cluster_id].append(history_sentences[idx])
    clustered_history_string +=history_sentences[idx]
    clustered_history_string += '\n'
  
  # Save the history clusters in a text file
  with open (f'data/clusters/history/history_clusters_{decade}', "w") as file :
     file.write(clustered_history_string)

  # Perform clustering on movies dataset
  movies_clusters = cluster_model.fit_predict(movies_embeddings.cpu().numpy())

  # Separate movies summaries into clusters
  clustered_movies_string=''
  clustered_movies = {i: [] for i in range(max(movies_clusters) + 1)}
  for idx, cluster_id in enumerate(movies_clusters):
    clustered_movies[cluster_id].append(movies_sentences[idx])
    clustered_movies_string+=movies_sentences[idx]
    clustered_movies_string += '\n'

  # Save the movie clusters in a text file
  with open (f'data/clusters/history/movies_clusters_{decade}', "w") as file :
     file.write(clustered_movies_string)

  # Compute average similarity per cluster
  max_similarities = []
  history_cluster_similarities = {i: [] for i in np.arange(len(clustered_history))+1}
  for history_cluster_id, sentences_history in clustered_history.items():
    history_cluster_emb = model.encode(sentences_history, convert_to_tensor=True).mean(dim=0)  # Average embedding for the cluster
    movies_similarities_list=[]
    for movie_cluster_id, sentences_movies in clustered_movies.items():
      movie_cluster_emb = model.encode(sentences_movies, convert_to_tensor=True).mean(dim=0)  # Average embedding for the cluster
      similarities = util.pytorch_cos_sim(history_cluster_emb, movie_cluster_emb).cpu().numpy()
      similarity = similarities[0][0]
      movies_similarities_list.append(similarity)
    print(movies_similarities_list)
    history_cluster_similarities[history_cluster_id+1]=movies_similarities_list  # Store similarities
    max_similarities.append(max(movies_similarities_list))  # Store max similarity for the bar plot


  # Initialize memory to keep the conversation context
  memory = ConversationBufferMemory()
  memory.clear()

  # Create a conversational chain with the model and memory
  conversation = ConversationChain(llm=llm, memory=memory)

  history_cluster_names_string=''
  history_cluster_names=[]

  for i in tqdm(range(5)):
      # clear memory
      memory = ConversationBufferMemory()
      memory.clear()

      text=clustered_history[i]

      response1 = conversation.run(text)

      if(i!=0) : other_names= ''
      else : other_names=history_cluster_names_string

      response3=conversation.run(f"From the most important key point of the previous text, give me an easily understandable one word topic, differerent from the topics : {other_names} and from the topic 'decade'. Give only 1 word, no addititonal text.")

      name=response3.replace('*','')
      name=response3.replace('.','')
      history_cluster_names.append(name)
      if i!=5 : name+=','
      history_cluster_names_string+=name
  print('history cluster names', history_cluster_names)

  movies_cluster_names_string=''
  movies_cluster_names=[]

  for i in tqdm(range(5)):
      # clear memory
      memory = ConversationBufferMemory()
      memory.clear()

      text=clustered_movies[i]

      response1 = conversation.run(text)

      if(i!=0) : other_names= ''
      else : other_names=movies_cluster_names_string

      response3=conversation.run(f"From the most important key point of the previous text, give me an easily understandable one word topic, differerent from the topics : {other_names} and from the topic 'decade' and from the topic 'movie'. Give only 1 word, no addititonal text.")

      name=response3.replace('*','')
      name=response3.replace('.','')
      movies_cluster_names.append(name)
      if i!=5 : name+=','
      movies_cluster_names_string+=name
  print('movies cluster names', movies_cluster_names)

  # Save bar plot data (Maximum similarity per cluster)
  bar_plot_data = pd.DataFrame({
      'Cluster ID': history_cluster_names,
      'Max Similarity': max_similarities
  })
  bar_plot_data.to_csv(f'data/max_similarity_plots_data/bar_plot_data_{decade}.csv', index=False)

  # Save heatmap data (similarity matrix)
  heatmap_data = pd.DataFrame(history_cluster_similarities)
  heatmap_data
  heatmap_data.index=movies_cluster_names
  heatmap_data.columns=history_cluster_names
  heatmap_data.to_csv(f'data/heatmap_plots_similarity_data/heatmap_data_{decade}.csv')




  0%|          | 0/11 [00:00<?, ?it/s]

[-0.038716316, -0.12739506, -0.06779581, -0.096319094, 0.13644566]
[0.21539037, 0.1435658, 0.008945765, 0.14175928, -0.06163284]
[0.42996696, 0.1353682, -0.12303482, 0.0877627, 0.2875343]
[0.24036545, 0.04255316, -0.06254479, 0.08022961, 0.4691099]
[0.23027168, 0.056057498, -0.017381324, -0.08695904, 0.17646024]


100%|██████████| 5/5 [01:39<00:00, 19.89s/it]


history cluster names ['Battery', 'Women', 'Opera', 'Flight', 'Transmit']


  9%|▉         | 1/11 [03:08<31:27, 188.70s/it]

...

[0.0028735679, 0.023149377, -0.017981615, -0.0012321249, 0.14796527]
[0.33114883, 0.14624764, 0.19231026, -0.050864257, 0.031371053]
[0.19904679, 0.46581408, 0.2045529, 0.13141143, 0.25120905]
[-0.12410659, 0.044492066, 0.020883618, -0.16577417, -0.124714985]
[0.24966395, 0.12719716, 0.3968816, -0.01932653, 0.027012346]


100%|██████████| 5/5 [02:50<00:00, 34.07s/it]


history cluster names ['Transfer', 'Election', 'Scandal', 'Trends', 'Sports']


100%|██████████| 11/11 [1:44:41<00:00, 571.08s/it] 

movies cluster names ['Trauma', 'Relationships', 'Adventure', 'Romance', 'Friendship']


In [ ]:
for decade in np.arange(1900,2010,10):
    with open('src/data/clusters_decades/clusters_history1900.txt',"r",encoding="utf-8") as file :
        clusters_1900=file.readlines()
        clusters_1900 = [cluster.replace("–", "-") for cluster in clusters_1900]
        clusters_1900 = [cluster.strip() for cluster in clusters_1900]

clustered_history = {i: [] for i in range(max(history_clusters) + 1)}
for idx, cluster_id in enumerate(history_clusters):
    clustered_history[cluster_id].append(history_sentences[idx])

In [ ]:
# Compute average similarity per cluster
topic_similarities = []
cluster_similarities = {i: [] for i in np.arange(len(clustered_history))+1}
print(cluster_similarities)
for cluster_id, sentences in clustered_history.items():
    cluster_emb = model.encode(sentences, convert_to_tensor=True).mean(dim=0)  # Average embedding for the cluster
    similarities = util.pytorch_cos_sim(cluster_emb, movies_embeddings).cpu().numpy()
    similarities_list = similarities[0].tolist()
    cluster_similarities[cluster_id+1]=similarities_list  # Store similarities
    topic_similarities.append(similarities.max())  # Store max similarity for the bar plot

# Save bar plot data (Maximum similarity per cluster)
bar_plot_data = pd.DataFrame({
    'Cluster ID': ['Science','War','Hero','Fashion','Film'],
    'Max Similarity': topic_similarities
})
bar_plot_data.to_csv('bar_plot_data.csv', index=False)
bar_plot_data.head()

In [1]:
from langchain_ollama import OllamaLLM
from langchain_ollama import OllamaLLM
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory
import numpy as np

llm = OllamaLLM(model="llama3.2")

In [6]:
with open('src/data/clusters_decades/clusters_history1900.txt',"r",encoding="utf-8") as file :
    clusters_1900=file.readlines()
    clusters_1900 = [cluster.replace("–", "-") for cluster in clusters_1900]
    clusters_1900 = [cluster.strip() for cluster in clusters_1900]
for i in clusters_1900 :
    print(i)

Reginald Fessenden  ofEast Bolto n, Quebec , Canada made what  appeared to be the first audio radio broadcasts of entertainment and music ever made to a general audience.[38][39]1901 - The first radio receiver  (successfully received a radio transmission).)[40] The Poldhu transmitter was a two-stage circuit.It is thefirst such device to play a series of gramophone records.[80] The V ictrola is also the first playback  machine containing an internal horn.[90]1906 - Sound radio broadcasting  was invented by Reginald Fessenden  and Lee De Forest .Fessenden and Ernst Alexanderson  developed ahigh-frequency alternator -transmitters, an improvement on an already existing device.The improved model operated at a transmittingfrequency of approximately 50 kHz, although with far less power than Fessenden's rotary-spark transmitters.The alternator-transmitterachieved the goal of transmitting quality audio signals, but the lack of any way to amplify the signals meant they were somewhat weak.OnDecem

In [8]:
from tqdm import tqdm

# Initialize memory to keep the conversation context
memory = ConversationBufferMemory()

# Create a conversational chain with the model and memory
conversation = ConversationChain(llm=llm, memory=memory)

cluster_names_string=''
cluster_names=[]
memory = ConversationBufferMemory()

for i in tqdm(range(5)):
    # clear memory
    memory.clear()

    text=clusters_1900[i]

    response1 = conversation.run(text)

    if(i!=0) : other_names= ''
    else : other_names=cluster_names_string

    response3=conversation.run(f"From the most important key point of the previous text, give me an easily understandable one word topic, differerent from the topics : {other_names} and from the topic 'decade'. Give only 1 word, no addititonal text.")

    name=response3.replace('*','')
    name=response3.replace('.','')
    cluster_names.append(name)
    if i!=5 : name+=','
    cluster_names_string+=name
print(cluster_names)

100%|██████████| 5/5 [01:58<00:00, 23.73s/it]

['Transistor', 'Automobile', 'Women', 'Invention', 'Innovation']


In [ ]:
movies_sentences = read_and_preprocess(f'src/data/movies_summaries/summaries_decade_{decade}.txt')
movies_embeddings = model.encode(movies_sentences, batch_size=32, convert_to_tensor=True)
# Cluster sentences from movies into sematic fields
clustered_movies = {i: [] for i in range(max(movies_clusters) + 1)}
for idx, cluster_id in enumerate(movies_clusters):
    clustered_movies[cluster_id].append(movies_sentences[idx])
clusters_movies = ''
for i in np.arange(5)+1:
    topic_text=''
    for j in clustered_movies[i-1]:
        topic_text+=j
    clusters_movies+=topic_text + '\n'

# save in file
with open(f'/content/drive/MyDrive/clusters_movies{decade}.txt', 'w') as f:
f.write(clusters_movies)
